# PAPC Final Runs — All Missing Pieces (~3.7h on A100 40GB)

Runs the four groups of missing work **in priority order**.
Upload `papc_B_results.json` and `papc_D_results.json` before starting.
Every result saves immediately to `papc_final_runs.json`.

| Priority | What | Why | Time |
|---|---|---|---|
| **1** | Gate-clamping ablation (dermamnist + pneumoniamnist) | Core mechanism: isolates aux-loss vs gate-integration | ~40 min |
| **2** | retinamnist SOTA table (3 conditions × 3 seeds) | Completes Session A | ~30 min |
| **3** | organamnist + organcmnist SOTA table | Completes Session A | ~90 min |
| **4** | organamnist + organcmnist Session D extended | Tight CIs | ~65 min |

**Note on gate-clamping (Priority 1):** `full_005` = use_pc, w=0.05, integration ON.
`clamped` = use_pc, w=0.05, integration OFF (gate force-zero). Vanilla already done.
Comparing the three reveals whether AUC damage / ECE shift comes from the auxiliary
loss or from the gate-integration step.

## 0. Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'medmnist','timm>=1.0','scikit-learn'])

import os, math, time, json, gc, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
import timm; from timm.data import Mixup
import medmnist; from medmnist import INFO
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory/1e9:.0f}GB')
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
HAS_BF16 = torch.cuda.is_bf16_supported() if device.type=='cuda' else False
AMP_DTYPE = torch.bfloat16 if HAS_BF16 else torch.float16
torch.manual_seed(0); np.random.seed(0)

MODEL_NAME = 'vit_base_patch16_224.augreg_in21k_ft_in1k'
IMG_SIZE   = 224
TIME_BUDGET_SEC = int(8.5 * 3600)
T0 = time.time()

for d in ('/teamspace/studios/this_studio','/kaggle/working','/content','.'):
    if os.path.isdir(d) and os.access(d, os.W_OK):
        OUTPUT_DIR = d; break
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Out:', OUTPUT_DIR, '| bf16:', HAS_BF16)

GPU: NVIDIA A100-SXM4-40GB  42GB
Out: /teamspace/studios/this_studio | bf16: True


## 1. Model (same as main notebook, cell-4 Mixup fix included)

In [2]:
class DiagonalSSM(nn.Module):
    def __init__(self, d_model, d_state=16, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        self.d_model = d_model; self.d_state = d_state
        a = torch.log(torch.linspace(1., float(d_state), d_state))
        self.A_log = nn.Parameter(a.repeat(d_model, 1))
        self.B  = nn.Parameter(torch.randn(d_model, d_state)*0.1)
        self.C  = nn.Parameter(torch.randn(d_model, d_state)*0.1)
        self.D  = nn.Parameter(torch.ones(d_model))
        dt = torch.rand(d_model)*(math.log(dt_max)-math.log(dt_min))+math.log(dt_min)
        self.dt_log = nn.Parameter(dt)
    def forward(self, x):
        B, L, D = x.shape
        dt   = torch.exp(self.dt_log.float())
        Abar = torch.exp(-torch.exp(self.A_log.float())*dt.unsqueeze(-1))
        Bbar = self.B.float()*dt.unsqueeze(-1)
        logA = torch.log(Abar.clamp(min=1e-38))
        k    = torch.arange(L, device=x.device, dtype=torch.float32)
        Apow = torch.exp(k.view(-1,1,1)*logA.unsqueeze(0))
        psi  = (Apow*(self.C.float()*Bbar).unsqueeze(0)).sum(-1).to(x.dtype)
        w    = psi.t().contiguous().unsqueeze(1).flip(-1)
        xp   = F.pad(x.transpose(1,2).contiguous(), (L-1,0))
        h    = F.conv1d(xp, w, groups=D)
        return (h + self.D.to(x.dtype).view(1,-1,1)*x.transpose(1,2)).transpose(1,2)

class PCLayer(nn.Module):
    def __init__(self, dim, d_state=16, has_error=True):
        super().__init__()
        self.has_error = has_error
        self.pred_norm = nn.LayerNorm(dim, eps=1e-6)
        self.predictor = DiagonalSSM(dim, d_state)
        if has_error:
            self.err_norm = nn.LayerNorm(dim, eps=1e-6)
            self.err_proj = nn.Linear(dim, dim)
            self.gate = nn.Parameter(torch.zeros(1))
    def predict(self, x): return self.predictor(self.pred_norm(x))
    def integrate(self, act, pred):
        return act + torch.tanh(self.gate)*self.err_proj(self.err_norm(act-pred))
    def gate_val(self):
        return 0. if not self.has_error else float(torch.tanh(self.gate).detach().cpu())

class PAPCViT(nn.Module):
    def __init__(self, model_name=MODEL_NAME, img_size=IMG_SIZE, num_classes=10,
                 in_chans=3, d_state=16, pred_loss_weight=0.05, drop_path=0.1,
                 pretrained=True, use_pc=True, adaptive=False, progressive=False,
                 clamp_gate=False, w_max=0.01, w_l2=1e-3):
        super().__init__()
        self.use_pc=use_pc; self.clamp_gate=clamp_gate
        self.pred_loss_weight=pred_loss_weight if use_pc else 0.
        self.adaptive=adaptive and use_pc; self.progressive=progressive
        self.w_max=w_max; self.w_l2=w_l2; self._step=0; self._total=1
        self.backbone=timm.create_model(model_name, pretrained=pretrained,
            img_size=img_size, num_classes=0, drop_path_rate=drop_path, in_chans=3)
        self.dim=self.backbone.embed_dim; self.depth=len(self.backbone.blocks)
        self.in_chans=in_chans; self.expand_gray=(in_chans==1)
        if use_pc:
            self.pc=nn.ModuleList([PCLayer(self.dim,d_state,has_error=(i>0))
                                   for i in range(self.depth)])
            if self.adaptive:
                iv=math.log(max(1e-8, math.exp(0.002/max(1e-8,w_max))-1.))
                self.log_w=nn.Parameter(torch.full((self.depth,),iv))
        else: self.pc=None
        self.head=nn.Linear(self.dim,num_classes)
        nn.init.trunc_normal_(self.head.weight,std=0.02); nn.init.zeros_(self.head.bias)

    def set_progress(self,s,t): self._step=s; self._total=t
    def _ps(self):
        if not self.progressive: return 1.
        return 0.5*(1.-math.cos(math.pi*self._step/max(1,self._total)))
    def _lw(self):
        return F.softplus(self.log_w)*self.w_max*self._ps() if self.adaptive else None

    def forward(self, x):
        if self.expand_gray: x=x.repeat(1,3,1,1)
        x=self.backbone.patch_embed(x); x=self.backbone._pos_embed(x)
        if hasattr(self.backbone,'patch_drop'): x=self.backbone.patch_drop(x)
        if hasattr(self.backbone,'norm_pre'): x=self.backbone.norm_pre(x)
        pl=x.new_zeros(())
        if self.use_pc:
            lw=self._lw(); prev=None
            for i,blk in enumerate(self.backbone.blocks):
                act=blk(x); pc=self.pc[i]
                if prev is not None and pc.has_error:
                    wi=lw[i] if lw is not None else self._ps()
                    pl=pl+wi*F.mse_loss(prev, act.detach())
                    if not self.clamp_gate: act=pc.integrate(act,prev)
                prev=pc.predict(act); x=act
            x=self.backbone.norm(x)
            if prev is not None:
                wl=lw[-1] if lw is not None else self._ps()
                pl=pl+wl*F.mse_loss(prev, x.detach())
            if self.adaptive: pl=pl+self.w_l2*(self._lw()**2).sum()
            elif not self.adaptive: pl=pl*self.pred_loss_weight
        else:
            for blk in self.backbone.blocks: x=blk(x)
            x=self.backbone.norm(x)
        return self.head(x[:,0]), pl

    def gate_values(self):
        return [] if not self.use_pc else [p.gate_val() for p in self.pc]
    def learned_weights(self):
        if not self.adaptive or not self.use_pc: return None
        with torch.no_grad(): return (F.softplus(self.log_w)*self.w_max).cpu().tolist()
    def param_groups(self, lr, hm=10., pm=5., dc=0.75):
        g=[]
        for i,blk in enumerate(self.backbone.blocks):
            g.append({'params':list(blk.parameters()),'lr':lr*dc**(self.depth-1-i)})
        early=list(self.backbone.patch_embed.parameters())
        for a in ('cls_token','pos_embed','reg_token'):
            p=getattr(self.backbone,a,None)
            if isinstance(p,nn.Parameter): early.append(p)
        g.append({'params':early,'lr':lr*dc**self.depth})
        g.append({'params':list(self.backbone.norm.parameters()),'lr':lr})
        if self.use_pc:
            pp=list(self.pc.parameters())
            if self.adaptive: pp.append(self.log_w)
            g.append({'params':pp,'lr':lr*pm})
        g.append({'params':list(self.head.parameters()),'lr':lr*hm})
        return [{'params':[p for p in x['params'] if isinstance(p,nn.Parameter)],'lr':x['lr']}
                for x in g if any(isinstance(p,nn.Parameter) for p in x['params'])]
print('Model ready')

Model ready


## 2. Data + metrics

In [3]:
IM=((0.485,0.456,0.406),(0.229,0.224,0.225))
def btf(sz,ic,tr):
    m,s=(IM[0],IM[1]) if ic==3 else ((sum(IM[0])/3,),(sum(IM[1])/3,))
    if tr:
        aug=T.RandAugment(2,9) if ic==3 else T.RandomAffine(10,(0.05,0.05))
        return T.Compose([T.Resize(int(sz*1.15),interpolation=3),
            T.RandomResizedCrop(sz,scale=(0.7,1.),interpolation=3),
            T.RandomHorizontalFlip(),aug,T.ToTensor(),T.Normalize(m,s),
            T.RandomErasing(p=0.25,scale=(0.02,0.15))])
    return T.Compose([T.Resize(int(sz*1.15),interpolation=3),
        T.CenterCrop(sz),T.ToTensor(),T.Normalize(m,s)])

class EMA:
    def __init__(self,m,dc=0.9995):
        self.dc=dc; self.sh={k:v.detach().clone() for k,v in m.state_dict().items()}
    @torch.no_grad()
    def update(self,m):
        for k,v in m.state_dict().items():
            if v.dtype.is_floating_point: self.sh[k].mul_(self.dc).add_(v.detach(),alpha=1-self.dc)
            else: self.sh[k]=v.detach().clone()
    def apply(self,m):
        bk={k:v.detach().clone() for k,v in m.state_dict().items()}
        m.load_state_dict(self.sh,strict=True); return bk
    def restore(self,m,bk): m.load_state_dict(bk,strict=True)

def metrics(yt,ys,task,nc):
    yt=np.asarray(yt); ys=np.nan_to_num(np.asarray(ys,dtype=np.float64))
    if task=='multi-label, binary-class':
        yc=np.clip(ys,0,1); acc=float(((yc>0.5).astype(int)==yt).mean())
        a=[roc_auc_score(yt[:,c],yc[:,c]) for c in range(nc) if 0<yt[:,c].sum()<len(yt)]
        return acc,float(np.mean(a)) if a else float('nan')
    y1=yt.squeeze().astype(np.int64)
    rs=ys.sum(1,keepdims=True) if ys.ndim>1 else None
    yn=ys/np.where(rs>0,rs,1) if rs is not None else ys
    yp=ys.argmax(1) if ys.ndim>1 else (ys>0.5).astype(int)
    acc=float((yp==y1).mean())
    if task=='binary-class' or nc==2:
        sc=yn if yn.ndim==1 else yn[:,1]
        try: return acc,float(roc_auc_score(y1,sc)) if len(np.unique(y1))>1 else (acc,float('nan'))
        except: return acc,float('nan')
    pc=[]
    for c in range(nc):
        yb=(y1==c).astype(int)
        if 0<yb.sum()<len(yb):
            try: pc.append(roc_auc_score(yb,yn[:,c]))
            except: pass
    return acc,float(np.mean(pc)) if pc else float('nan')

def ece_fn(yt,ys,nb=15):
    y1=np.asarray(yt).squeeze().astype(int)
    ys2=np.nan_to_num(np.asarray(ys,dtype=np.float64))
    if ys2.ndim==1: ys2=np.stack([1-ys2,ys2],1)
    conf=ys2.max(1); pred=ys2.argmax(1); ok=(pred==y1).astype(float)
    bins=np.linspace(0,1,nb+1); e=0.
    for i in range(nb):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()>0: e+=m.mean()*abs(ok[m].mean()-conf[m].mean())
    return float(e)

def load_med(flag,sz=IMG_SIZE):
    info=INFO[flag]; DC=getattr(medmnist,info['python_class'])
    ic=info['n_channels']; task=info['task']
    nc=len(info['label']) if isinstance(info['label'],dict) else int(info['label'])
    ssz=sz if sz in (28,64,128,224) else 224
    root=os.path.join(OUTPUT_DIR,'medmnist_data'); os.makedirs(root,exist_ok=True)
    tr=DC(split='train',transform=btf(sz,ic,True), download=True,size=ssz,root=root)
    va=DC(split='val',  transform=btf(sz,ic,False),download=True,size=ssz,root=root)
    te=DC(split='test', transform=btf(sz,ic,False),download=True,size=ssz,root=root)
    return tr,va,te,task,nc,ic
print('Data ready')

Data ready


## 3. Training driver

In [4]:
def train_eval(mkw, tr, va, te, task, nc, ic, ep, bs, lr, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    if device.type=='cuda': torch.cuda.manual_seed_all(seed)
    is_ml=task=='multi-label, binary-class'
    model=PAPCViT(MODEL_NAME,IMG_SIZE,nc,ic,pretrained=True,**mkw).to(device)
    nw=min(10,os.cpu_count() or 4)
    kw=dict(num_workers=nw,pin_memory=True,persistent_workers=(nw>0))
    tl=DataLoader(tr,bs,shuffle=True,drop_last=True,**kw)
    el=DataLoader(te,bs*2,**kw)
    mfn=Mixup(mixup_alpha=0.2,cutmix_alpha=1.0,prob=1.0,switch_prob=0.5,
              mode='batch',label_smoothing=0.1,num_classes=nc
             ) if (not is_ml and nc>2) else None
    opt=torch.optim.AdamW(model.param_groups(lr),weight_decay=0.05,betas=(0.9,0.95))
    Tt=len(tl)*ep; W=max(1,int(Tt*0.05))
    sch=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:(
        s/W if s<W else 0.5*(1+math.cos(math.pi*(s-W)/max(1,Tt-W)))))
    ema=EMA(model); t0=time.time(); gs=0
    for _ in range(ep):
        model.train()
        for x,y in tl:
            model.set_progress(gs,Tt)
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            yl=y.long().squeeze(-1) if y.ndim>1 else y.long()
            if mfn and not is_ml: x,tgt=mfn(x,yl)
            elif is_ml: tgt=y.float()
            else: tgt=F.one_hot(yl,nc).float()*0.9+0.1/nc
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda',dtype=AMP_DTYPE,
                                    enabled=(device.type=='cuda')):
                lo,pl=model(x)
                main=(F.binary_cross_entropy_with_logits(lo,tgt) if is_ml
                      else -(tgt*F.log_softmax(lo,-1)).sum(-1).mean())
                loss=main+pl
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step(); sch.step(); ema.update(model); gs+=1
    bk=ema.apply(model); model.eval(); ys_l,ss_l=[],[]
    with torch.no_grad():
        for x,y in el:
            x=x.to(device,non_blocking=True)
            with torch.amp.autocast(device_type='cuda',dtype=AMP_DTYPE,
                                    enabled=(device.type=='cuda')):
                la,_=model(x); lb,_=model(torch.flip(x,[-1]))
            avg=(la+lb)/2
            s=torch.sigmoid(avg) if is_ml else F.softmax(avg,dim=-1)
            ys_l.append(y.cpu().numpy()); ss_l.append(s.float().cpu().numpy())
    ema.restore(model,bk)
    ya=np.concatenate(ys_l); sa=np.concatenate(ss_l)
    acc,auc=metrics(ya,sa,task,nc); ecv=ece_fn(ya,sa)
    gv=float(np.mean(np.abs(model.gate_values()))) if mkw.get('use_pc') else 0.
    lw=model.learned_weights()
    del model,ema,opt,sch,tl,el
    if device.type=='cuda': torch.cuda.empty_cache(); gc.collect()
    return dict(acc=acc,auc=auc,ece=ecv,gate=gv,lw=lw,t_min=(time.time()-t0)/60)
print('train_eval ready')

train_eval ready


## 4. Load existing results + init output file

In [5]:
def find_json(name):
    for d in (OUTPUT_DIR,'/teamspace/uploads/','.'):
        p=os.path.join(d,name)
        if os.path.exists(p): return p
    return None

B_data = json.load(open(find_json('papc_B_results.json'))) if find_json('papc_B_results.json') else {}
D_data = json.load(open(find_json('papc_D_results.json'))) if find_json('papc_D_results.json') else {}
print('Loaded B:', list(B_data.keys()))
print('Loaded D:', list(D_data.keys()))

OUT = os.path.join(OUTPUT_DIR,'papc_final_runs.json')
if os.path.exists(OUT):
    FINAL = json.load(open(OUT))
    print('Resumed existing output:', OUT)
else:
    FINAL = {'clamp':{}, 'sota':{}, 'extended':{}}

def save():
    json.dump(FINAL, open(OUT,'w'), indent=2,
              default=lambda o: float(o) if hasattr(o,'item') else str(o))

def tl(): return TIME_BUDGET_SEC-(time.time()-T0)

def already_have(section, flag, cname, needed):
    # check FINAL and existing B/D results
    in_final = FINAL.get(section,{}).get(flag,{}).get(cname,[])
    if section == 'clamp':
        in_existing = B_data.get('clamp',{}).get(flag,{}).get(cname,[])
    elif section == 'sota':
        in_existing = B_data.get('sota',{}).get(flag,{}).get(cname,[])
    elif section == 'extended':
        in_existing = D_data.get('extended',{}).get(flag,{}).get(cname,[])
    else:
        in_existing = []
    total = len(in_final) + len(in_existing)
    return total, needed-total   # (have, need)

print('Ready. tl={:.1f}h'.format(tl()/3600))

Loaded B: []
Loaded D: []
Ready. tl=8.5h


## Priority 1 — Gate-clamping ablation
**Most critical for the paper.** Compares:
- `vanilla`: no PC
- `full_005`: PC w=0.05, integration ON (= same as fixed_0.05 in Session B ablation)
- `clamped`: PC w=0.05, integration OFF (gate force-zeroed)

If `full_005 AUC < vanilla` but `clamped AUC ≈ full_005`: the gate integration does not explain the AUC harm — the aux loss itself does.
If `clamped AUC > full_005`: the integration makes things worse than aux loss alone.
Either way, this isolates the mechanism and completes the calibration-lever story.

In [6]:
CLAMP_DS = {
    'dermamnist':     dict(ep=22, bs=96,  lr=1e-4),
    'pneumoniamnist': dict(ep=22, bs=96,  lr=1e-4),
}
CLAMP_CONDS = [
    ('vanilla',  dict(use_pc=False)),
    ('full_005', dict(use_pc=True, pred_loss_weight=0.05)),
    ('clamped',  dict(use_pc=True, pred_loss_weight=0.05, clamp_gate=True)),
]

print('='*60)
print('PRIORITY 1: Gate-clamping ablation')
print('='*60)

for flag, cfg in CLAMP_DS.items():
    if tl()<600: print(f'TIME: skip {flag}'); break
    try: tr,va,te,task,nc,ic=load_med(flag)
    except Exception as e: print(f'{flag}: {e}'); continue
    print(f'\n{flag} (n={len(tr)}, task={task})')
    FINAL['clamp'].setdefault(flag,{'n':len(tr)})

    for cname, mkw in CLAMP_CONDS:
        have, need = already_have('clamp', flag, cname, 3)
        if need <= 0:
            # Show existing numbers
            all_r = (B_data.get('clamp',{}).get(flag,{}).get(cname,[]) +
                     FINAL['clamp'][flag].get(cname,[]))
            if all_r:
                aucs=[r['auc'] for r in all_r]; eces=[r['ece'] for r in all_r]
                print(f'  {cname:10s}: ALREADY DONE  AUC={np.mean(aucs):.4f}  ECE={np.mean(eces):.4f}')
            continue

        FINAL['clamp'][flag].setdefault(cname, [])
        for s in range(have, have+need):
            if tl()<300: print('  TIME: stop'); break
            r = train_eval(mkw, tr, va, te, task, nc, ic, cfg['ep'], cfg['bs'], cfg['lr'], s)
            FINAL['clamp'][flag][cname].append(r)
            save()
            print(f'  {cname:10s} s{s}: AUC={r["auc"]:.4f}  ACC={r["acc"]:.4f}  '
                  f'ECE={r["ece"]:.4f}  gate={r["gate"]:.3f}  ({r["t_min"]:.1f}m)')

    # Summary for this dataset
    print(f'  --- {flag} clamp summary ---')
    for cname, _ in CLAMP_CONDS:
        all_r = (B_data.get('clamp',{}).get(flag,{}).get(cname,[]) +
                 B_data.get('ablation',{}).get(flag,{}).get(
                     'fixed_0.05' if cname=='full_005' else cname, []) +
                 FINAL['clamp'].get(flag,{}).get(cname,[]))
        # also check ablation for full_005 (= fixed_0.05)
        if cname == 'full_005' and not all_r:
            all_r = B_data.get('ablation',{}).get(flag,{}).get('fixed_0.05',[])
        aucs=[r['auc'] for r in all_r if isinstance(r,dict)]
        eces=[r['ece'] for r in all_r if isinstance(r,dict)]
        if aucs:
            print(f'  {cname:10s}: n={len(aucs)}  AUC={np.mean(aucs):.4f}  ECE={np.mean(eces):.4f}')
print('\nPriority 1 done. Elapsed: {:.1f}min'.format((time.time()-T0)/60))

PRIORITY 1: Gate-clamping ablation


100%|██████████| 1.09G/1.09G [00:50<00:00, 21.8MB/s]



dermamnist (n=7007, task=multi-class)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  vanilla    s0: AUC=0.9821  ACC=0.8778  ECE=0.0720  gate=0.000  (3.0m)


  vanilla    s1: AUC=0.9812  ACC=0.8733  ECE=0.0765  gate=0.000  (3.0m)
  vanilla    s2: AUC=0.9822  ACC=0.8698  ECE=0.0948  gate=0.000  (3.0m)
  full_005   s0: AUC=0.9751  ACC=0.8090  ECE=0.0645  gate=0.069  (9.7m)
  full_005   s1: AUC=0.9569  ACC=0.3930  ECE=0.2385  gate=0.055  (9.7m)
  full_005   s2: AUC=0.9591  ACC=0.4549  ECE=0.2429  gate=0.065  (9.7m)
  clamped    s0: AUC=0.9751  ACC=0.8030  ECE=0.0418  gate=0.000  (9.1m)
  clamped    s1: AUC=0.9592  ACC=0.4798  ECE=0.2075  gate=0.000  (9.1m)
  clamped    s2: AUC=0.9556  ACC=0.4893  ECE=0.2259  gate=0.000  (9.1m)
  --- dermamnist clamp summary ---
  vanilla   : n=3  AUC=0.9818  ECE=0.0811
  full_005  : n=3  AUC=0.9637  ECE=0.1820
  clamped   : n=3  AUC=0.9633  ECE=0.1584


100%|██████████| 214M/214M [00:15<00:00, 13.4MB/s] 



pneumoniamnist (n=4708, task=binary-class)
  vanilla    s0: AUC=0.9923  ACC=0.8830  ECE=0.0394  gate=0.000  (1.9m)
  vanilla    s1: AUC=0.9908  ACC=0.9583  ECE=0.0828  gate=0.000  (1.9m)
  vanilla    s2: AUC=0.9925  ACC=0.9423  ECE=0.0453  gate=0.000  (1.9m)
  full_005   s0: AUC=0.9915  ACC=0.9407  ECE=0.0124  gate=0.058  (6.5m)
  full_005   s1: AUC=0.9878  ACC=0.9263  ECE=0.0168  gate=0.064  (6.5m)
  full_005   s2: AUC=0.9883  ACC=0.9119  ECE=0.0254  gate=0.060  (6.5m)
  clamped    s0: AUC=0.9905  ACC=0.9551  ECE=0.0190  gate=0.000  (6.1m)
  clamped    s1: AUC=0.9890  ACC=0.9503  ECE=0.0162  gate=0.000  (6.1m)
  clamped    s2: AUC=0.9894  ACC=0.9567  ECE=0.0152  gate=0.000  (6.1m)
  --- pneumoniamnist clamp summary ---
  vanilla   : n=3  AUC=0.9919  ECE=0.0558
  full_005  : n=3  AUC=0.9892  ECE=0.0182
  clamped   : n=3  AUC=0.9896  ECE=0.0168

Priority 1 done. Elapsed: 111.1min


## Priority 2 — retinamnist: Session A SOTA table

In [7]:
SOTA_CONDS = [
    ('vanilla', dict(use_pc=False)),
    ('hp_0.05', dict(use_pc=True, pred_loss_weight=0.05)),
    ('PAPC',    dict(use_pc=True, adaptive=True, progressive=True, w_max=0.01, w_l2=1e-3)),
]
SOTA_MISSING = {
    'retinamnist': dict(ep=35, bs=32, lr=5e-5),
}

print('='*60)
print('PRIORITY 2: retinamnist SOTA table')
print('='*60)

for flag, cfg in SOTA_MISSING.items():
    if tl()<600: print(f'TIME: skip {flag}'); break
    try: tr,va,te,task,nc,ic=load_med(flag)
    except Exception as e: print(f'{flag}: {e}'); continue
    print(f'\n{flag} (n={len(tr)}, task={task}, K={nc})')
    FINAL['sota'].setdefault(flag,{'n':len(tr)})

    for cname, mkw in SOTA_CONDS:
        have, need = already_have('sota', flag, cname, 3)
        if need <= 0:
            all_r = FINAL['sota'][flag].get(cname,[])
            if all_r: print(f'  {cname:10s}: DONE  AUC={np.mean([r["auc"] for r in all_r]):.4f}')
            continue
        FINAL['sota'][flag].setdefault(cname,[])
        for s in range(have, have+need):
            if tl()<300: break
            r=train_eval(mkw,tr,va,te,task,nc,ic,cfg['ep'],cfg['bs'],cfg['lr'],s)
            FINAL['sota'][flag][cname].append(r); save()
            print(f'  {cname:10s} s{s}: AUC={r["auc"]:.4f}  ECE={r["ece"]:.4f}  ({r["t_min"]:.1f}m)')

    # Summary
    for cname,_ in SOTA_CONDS:
        runs=FINAL['sota'].get(flag,{}).get(cname,[])
        if runs:
            aucs=[r['auc'] for r in runs]
            print(f'  {cname:10s} final: {len(aucs)} seeds  AUC={np.mean(aucs):.4f}+-{np.std(aucs):.4f}')
print('\nPriority 2 done. Elapsed: {:.1f}min'.format((time.time()-T0)/60))

PRIORITY 2: retinamnist SOTA table


100%|██████████| 128M/128M [00:08<00:00, 14.9MB/s] 



retinamnist (n=1080, task=ordinal-regression, K=5)
  vanilla    s0: AUC=0.8672  ECE=0.1025  (1.0m)
  vanilla    s1: AUC=0.8684  ECE=0.0813  (1.0m)
  vanilla    s2: AUC=0.8342  ECE=0.1120  (1.0m)
  hp_0.05    s0: AUC=0.8624  ECE=0.1562  (2.8m)
  hp_0.05    s1: AUC=0.8652  ECE=0.0709  (2.8m)
  hp_0.05    s2: AUC=0.8559  ECE=0.0896  (2.8m)
  PAPC       s0: AUC=0.8679  ECE=0.1025  (2.8m)
  PAPC       s1: AUC=0.8588  ECE=0.0892  (2.8m)
  PAPC       s2: AUC=0.8541  ECE=0.0731  (2.8m)
  vanilla    final: 3 seeds  AUC=0.8566+-0.0158
  hp_0.05    final: 3 seeds  AUC=0.8612+-0.0039
  PAPC       final: 3 seeds  AUC=0.8602+-0.0057

Priority 2 done. Elapsed: 131.2min


## Priority 3 — organamnist + organcmnist: Session A SOTA table

In [8]:
ORGAN_CFG = {
    'organamnist': dict(ep=10, bs=128, lr=1e-4),
    'organcmnist': dict(ep=12, bs=128, lr=1e-4),
}

print('='*60)
print('PRIORITY 3: organamnist + organcmnist SOTA table')
print('='*60)

for flag, cfg in ORGAN_CFG.items():
    if tl()<600: print(f'TIME: skip {flag}'); break
    try: tr,va,te,task,nc,ic=load_med(flag)
    except Exception as e: print(f'{flag}: {e}'); continue
    print(f'\n{flag} (n={len(tr)})')
    FINAL['sota'].setdefault(flag,{'n':len(tr)})

    for cname, mkw in SOTA_CONDS:
        have, need = already_have('sota', flag, cname, 3)
        if need <= 0: continue
        FINAL['sota'][flag].setdefault(cname,[])
        for s in range(have, have+need):
            if tl()<300: break
            r=train_eval(mkw,tr,va,te,task,nc,ic,cfg['ep'],cfg['bs'],cfg['lr'],s)
            FINAL['sota'][flag][cname].append(r); save()
            print(f'  {cname:10s} s{s}: AUC={r["auc"]:.4f}  ECE={r["ece"]:.4f}  ({r["t_min"]:.1f}m)')
print('\nPriority 3 done. Elapsed: {:.1f}min'.format((time.time()-T0)/60))

PRIORITY 3: organamnist + organcmnist SOTA table


100%|██████████| 1.80G/1.80G [01:08<00:00, 26.3MB/s] 



organamnist (n=34561)
  vanilla    s0: AUC=0.9950  ECE=0.0476  (6.0m)
  vanilla    s1: AUC=0.9973  ECE=0.0569  (6.0m)
  vanilla    s2: AUC=0.9964  ECE=0.0516  (6.0m)
  hp_0.05    s0: AUC=0.9967  ECE=0.1023  (21.4m)
  hp_0.05    s1: AUC=0.9949  ECE=0.0658  (21.4m)
  hp_0.05    s2: AUC=0.9971  ECE=0.0494  (21.4m)
  PAPC       s0: AUC=0.9968  ECE=0.0637  (21.4m)
  PAPC       s1: AUC=0.9962  ECE=0.0321  (21.4m)
  PAPC       s2: AUC=0.9942  ECE=0.0383  (21.4m)


100%|██████████| 760M/760M [00:40<00:00, 18.9MB/s] 



organcmnist (n=12975)
  vanilla    s0: AUC=0.9845  ECE=0.1725  (2.7m)
  vanilla    s1: AUC=0.9886  ECE=0.1381  (2.8m)
  vanilla    s2: AUC=0.9894  ECE=0.1574  (2.8m)
  hp_0.05    s0: AUC=0.9874  ECE=0.0677  (9.7m)
  hp_0.05    s1: AUC=0.9882  ECE=0.1235  (9.7m)
  hp_0.05    s2: AUC=0.9818  ECE=0.0359  (9.7m)
  PAPC       s0: AUC=0.9890  ECE=0.1668  (9.7m)
  PAPC       s1: AUC=0.9904  ECE=0.1156  (9.7m)
  PAPC       s2: AUC=0.9850  ECE=0.1748  (9.7m)

Priority 3 done. Elapsed: 346.9min


## Priority 4 — Session D extended (5-seed tight CIs)

In [9]:
EXT_CFG = {
    'organamnist': dict(ep=10, bs=128, lr=1e-4),
    'organcmnist': dict(ep=12, bs=128, lr=1e-4),
}
EXT_CONDS = [
    ('vanilla', dict(use_pc=False)),
    ('PAPC',    dict(use_pc=True, adaptive=True, progressive=True, w_max=0.01, w_l2=1e-3)),
]

print('='*60)
print('PRIORITY 4: Session D extended table')
print('='*60)

for flag, cfg in EXT_CFG.items():
    if tl()<600: print(f'TIME: skip {flag}'); break
    try: tr,va,te,task,nc,ic=load_med(flag)
    except Exception as e: print(f'{flag}: {e}'); continue
    print(f'\n{flag} (n={len(tr)})')
    FINAL['extended'].setdefault(flag,{'n':len(tr)})

    for cname, mkw in EXT_CONDS:
        have, need = already_have('extended', flag, cname, 5)
        if need <= 0: continue
        FINAL['extended'][flag].setdefault(cname,[])
        for s in range(have, have+need):
            if tl()<300: break
            r=train_eval(mkw,tr,va,te,task,nc,ic,cfg['ep'],cfg['bs'],cfg['lr'],s)
            FINAL['extended'][flag][cname].append(r); save()
            print(f'  {cname:10s} s{s}: AUC={r["auc"]:.4f}  ({r["t_min"]:.1f}m)')
print('\nPriority 4 done. Elapsed: {:.1f}min'.format((time.time()-T0)/60))

PRIORITY 4: Session D extended table

organamnist (n=34561)
  vanilla    s0: AUC=0.9950  (6.0m)
  vanilla    s1: AUC=0.9973  (6.0m)
  vanilla    s2: AUC=0.9964  (6.0m)
  vanilla    s3: AUC=0.9961  (6.0m)
  vanilla    s4: AUC=0.9973  (6.0m)
  PAPC       s0: AUC=0.9968  (21.4m)
  PAPC       s1: AUC=0.9962  (21.4m)
  PAPC       s2: AUC=0.9942  (21.4m)
  PAPC       s3: AUC=0.9969  (21.4m)
  PAPC       s4: AUC=0.9967  (21.4m)

organcmnist (n=12975)
  vanilla    s0: AUC=0.9845  (2.8m)
  vanilla    s1: AUC=0.9886  (2.8m)
  vanilla    s2: AUC=0.9894  (2.8m)
  vanilla    s3: AUC=0.9889  (2.8m)
  vanilla    s4: AUC=0.9871  (2.7m)
  PAPC       s0: AUC=0.9890  (9.7m)

Priority 4 done. Elapsed: 508.5min


## Final summary

In [10]:
save()
print(f'\nSaved: {OUT}')
print(f'Total elapsed: {(time.time()-T0)/60:.1f} min\n')

# Gate-clamping results — the key paper table
print('GATE-CLAMPING RESULTS (key for mechanism section)')
print(f'{"Dataset":18s}  {"Condition":10s}  {"AUC":>8s}  {"ECE":>8s}  {"Interpretation"}')
PUB={'dermamnist':0.937,'pneumoniamnist':0.995,'retinamnist':0.773,
     'organamnist':0.998,'organcmnist':0.997}

for flag in ('dermamnist','pneumoniamnist'):
    fd_b_abl = B_data.get('ablation',{}).get(flag,{})
    fd_b_clm = B_data.get('clamp',{}).get(flag,{})
    fd_fin   = FINAL.get('clamp',{}).get(flag,{})
    for cname in ('vanilla','full_005','clamped'):
        src_name = 'fixed_0.05' if cname=='full_005' else cname
        runs = (fd_b_clm.get(cname,[]) + fd_b_abl.get(src_name,[]) +
                fd_fin.get(cname,[]))
        if not runs: continue
        aucs=[r['auc'] for r in runs if isinstance(r,dict)]
        eces=[r['ece'] for r in runs if isinstance(r,dict)]
        tag = ('baseline' if cname=='vanilla'
               else ('aux+integration' if cname=='full_005' else 'aux only'))
        print(f'  {flag:18s}  {cname:10s}  {np.mean(aucs):>8.4f}  '
              f'{np.mean(eces):>8.4f}  {tag}')

# SOTA table summary
print('\nSOTA TABLE SUMMARY (all completed datasets)')
print(f'{"Dataset":18s}  {"n":>6}  {"Vanilla":>8}  {"hp.05":>8}  {"PAPC":>8}  {"Best Pub":>8}  {"Win?"}')
all_sota = {}
for src in (B_data.get('sota',{}), FINAL.get('sota',{})):
    for flag, fd in src.items():
        all_sota.setdefault(flag, {'n': fd.get('n','?')})
        for c in ('vanilla','hp_0.05','PAPC'):
            all_sota[flag].setdefault(c, [])
            all_sota[flag][c].extend(fd.get(c,[]))

# Also from Session A original
import os as _os
a_path = find_json('papc_A_results.json')
if a_path:
    A_data = json.load(open(a_path))
    for flag, fd in A_data.get('sota',{}).items():
        all_sota.setdefault(flag,{'n':fd.get('n','?')})
        for c in ('vanilla','hp_0.05','PAPC'):
            all_sota[flag].setdefault(c,[])
            all_sota[flag][c].extend(fd.get(c,[]))

for flag, fd in sorted(all_sota.items(), key=lambda x: x[1].get('n',0) or 0, reverse=True):
    va = np.mean([r['auc'] for r in fd.get('vanilla',[]) if isinstance(r,dict)]) if fd.get('vanilla') else float('nan')
    hp = np.mean([r['auc'] for r in fd.get('hp_0.05',[]) if isinstance(r,dict)]) if fd.get('hp_0.05') else float('nan')
    pa = np.mean([r['auc'] for r in fd.get('PAPC',[]) if isinstance(r,dict)]) if fd.get('PAPC') else float('nan')
    pub = PUB.get(flag, float('nan'))
    win = 'VA WIN' if va > pub else ('PAPC WIN' if pa > pub else '')
    n = fd.get('n','?')
    print(f'  {flag:18s}  {str(n):>6}  {va:>8.4f}  {hp:>8.4f}  {pa:>8.4f}  {pub:>8.3f}  {win}')
print(f'\nDownload papc_final_runs.json and send it back — ready to write the paper.')


Saved: /teamspace/studios/this_studio/papc_final_runs.json
Total elapsed: 508.5 min

GATE-CLAMPING RESULTS (key for mechanism section)
Dataset             Condition        AUC       ECE  Interpretation
  dermamnist          vanilla       0.9818    0.0811  baseline
  dermamnist          full_005      0.9637    0.1820  aux+integration
  dermamnist          clamped       0.9633    0.1584  aux only
  pneumoniamnist      vanilla       0.9919    0.0558  baseline
  pneumoniamnist      full_005      0.9892    0.0182  aux+integration
  pneumoniamnist      clamped       0.9896    0.0168  aux only

SOTA TABLE SUMMARY (all completed datasets)
Dataset                  n   Vanilla     hp.05      PAPC  Best Pub  Win?
  organamnist          34561    0.9962    0.9962    0.9957     0.998  
  organcmnist          12975    0.9875    0.9858    0.9881     0.997  
  retinamnist           1080    0.8566    0.8612    0.8602     0.773  VA WIN

Download papc_final_runs.json and send it back — ready to write the